# Full Scale Model Training
A Jupyter notebook for fetching bulk PDB data safely via RSYNC and training the full-scale structured graph model using GPU acceleration.

## 1. Setup Environment and Hardware Check
Import required libraries, define folder structures, and verify CUDA availability to ensure the GPU is ready for the training phase.

In [ ]:
import os
import subprocess
import sys
import torch

# Ensure we operate from the project root and don't recursively jump directories
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)
    
print(f"Working Directory: {os.getcwd()}")

# Hardware check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Compute Device: {device}")

## 2. CPU Phase: Bulk Data Download via RSYNC

If you try to hit the RCSB REST API with thousands of concurrent requests, you will trigger their automated defenses (HTTP 429). The official recommendation for HPC/SLURM environments is to use their dedicated `rsync` server.

This cell downloads their official `rsyncPDB.sh` script and executes it to bulk download the dataset to local storage.

In [ ]:
data_dir = os.path.join(project_root, "data", "pdb_archive")
os.makedirs(data_dir, exist_ok=True)

rsync_script_url = "https://cdn.rcsb.org/rcsb-pdb/general_information/news_publications/rsyncPDB.sh"
script_path = os.path.join(project_root, "scripts", "rsyncPDB.sh")
os.makedirs(os.path.dirname(script_path), exist_ok=True)

print("Downloading rsyncPDB.sh...")
subprocess.run(["curl", "-s", "-O", rsync_script_url], cwd=os.path.dirname(script_path), check=True)
subprocess.run(["chmod", "+x", script_path], check=True)

print(f"Target Data Directory: {data_dir}")
print("Run the following script to begin the RSYNC mirroring:")
print(f"{script_path} {data_dir}")

# Uncomment to initiate exactly that ~100GB mirror process
# subprocess.run([script_path, data_dir], check=True)

## 3. CPU Phase: Data Preprocessing and Caching
Extract, clean, and pre-process the locally mirrored PDB files into Torch-compatible graph representations using multiprocessing on the CPU.

In [ ]:
# A placeholder configuration cell for dataset graph pre-generation using Multiple CPU workers
print("Proceeding to graph generation using dataset.py configurations...")
# Example command:
# !python scripts/preprocess.py --data_dir data/pdb_archive --out_dir data/processed --num_workers 16

## 4. GPU Phase: Model, Loss, and Optimizer Initialization
Initialize the heterogeneous graph model, defining the loss functions and optimizer, and move the model parameters to the GPU device.

In [ ]:
from utils.model_utils import HeteroStruct2Seq

# Instantiate the newly decoupled graph model
model = HeteroStruct2Seq(
    node_dim=128, 
    edge_dim=64, 
    num_layers=4, 
    vocab_size=21
)

model = model.to(device)
print("Model initialized and moved to:", device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss(ignore_index=0) # 0 = PAD

## 5. GPU Phase: Full-Scale Training Loop
Instantiate the PyTorch DataLoader to read from the local dataset and execute the forward and backward passes on the GPU to train the model.

In [ ]:
train_cmd = [
    "python", "scripts/train.py",
    "--data_dir", "data/pdb_archive",
    "--batch_size", "8",
    "--num_workers", "8", # CRITICAL: Maximize CPU-to-GPU throughput
    "--pin_memory",       # CRITICAL: Fast Host-to-Device transfer
    "--epochs", "50",
    "--lr", "1e-4",
    "--checkpoint_interval", "5000",
    "--log_interval", "100"
]

print(f"Ready to execute final training command:\n{' '.join(train_cmd)}")

# Uncomment to actually run the background training subprocess
# subprocess.run(train_cmd)